# Baseline characteristics
Summarizes the VUmc, LUMC, and combined enrolled cohorts.

In [ ]:
# Parameters

project_root = "../"
data_dir = "../data"

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import pandas as pd

from algo_vs_doc import (
    baseline_descriptives,
    outcome_descriptives,
    prediction_data,
    specialties,
    subgroup_descriptives,
)
from algo_vs_doc.scoring.final_assembly import apply_enrolment_funnel


outputs_folder = os.path.join(project_root, "results", "summaries")
report_cohort = "Combined"
report_outputs_folder = os.path.join(outputs_folder, report_cohort)
os.makedirs(report_outputs_folder, exist_ok=True)

loaded_cohorts = prediction_data.load_cohorts(data_dir, include_combined=False)
enrolled_cohorts = {
    name: apply_enrolment_funnel(frame)[0] for name, frame in loaded_cohorts.items()
}
baseline_cohorts = {
    "Combined": pd.concat(enrolled_cohorts.values(), ignore_index=True),
    "VUmc": enrolled_cohorts["VUmc"],
    "LUMC": enrolled_cohorts["LUMC"],
}

In [ ]:
pd.DataFrame(
    {
        name: specialties.labels(frame["specialty"]).value_counts()
        for name, frame in baseline_cohorts.items()
    }
).fillna(0).astype(int)

In [ ]:
pd.concat(
    {name: frame.head() for name, frame in baseline_cohorts.items()},
    names=["cohort"],
)

## Setttings and Data

In [ ]:
lumc = baseline_cohorts["LUMC"]
vumc = baseline_cohorts["VUmc"]

## Load Data

In [ ]:
pd.DataFrame(
    {
        name: {"icu_stays": len(frame), "columns": frame.shape[1]}
        for name, frame in baseline_cohorts.items()
    }
).T

## LUMC baseline characteristics

In [ ]:
len(lumc)

In [ ]:
lumc["icu_los"].describe()

In [ ]:
lumc["age_at_admission"].describe()

In [ ]:
lumc["sex"].fillna("unknown").value_counts()

In [ ]:
specialties.labels(lumc["specialty"]).fillna("Unknown").value_counts()

In [ ]:
lumc["icu_los"].describe()

## VUMC baseline characteristics

In [ ]:
len(vumc)

In [ ]:
vumc["age_at_admission"].describe()

In [ ]:
vumc["sex"].fillna("unknown").value_counts()

In [ ]:
specialties.labels(vumc["specialty"]).fillna("Unknown").value_counts()

In [ ]:
vumc["icu_los"].describe()

## Create Baseline Characteristics table

In [ ]:
baseline = baseline_descriptives.baseline_characteristics_table(baseline_cohorts)
baseline.to_csv(os.path.join(report_outputs_folder, "baseline_characteristics.csv"))
baseline

## Mutually exclusive seven-day event types

This separates death, ICU readmission, HDU readmission, and overlapping events rather than reporting only the composite endpoint.


In [ ]:
event_types = outcome_descriptives.event_counts_by_cohort(baseline_cohorts)
event_types.to_csv(
    os.path.join(report_outputs_folder, "seven_day_event_types.csv"), index=False
)
event_types

## Characteristics of patients with and without an event

Because the number of events is small, these are descriptive summaries rather than powered subgroup comparisons. No patient or stay identifiers are exported.


In [ ]:
event_characteristics = baseline_descriptives.outcome_group_summaries(baseline_cohorts)
event_characteristics.to_csv(
    os.path.join(report_outputs_folder, "event_vs_no_event_characteristics.csv"),
    index=False,
)
event_characteristics

## Clinical subgroup descriptives

Outcome counts are shown by admitting specialty, admission type, and explicit ICU length-of-stay bands. These analyses are descriptive: sparse cells are not used to claim subgroup-specific performance.


In [ ]:
clinical_subgroups = subgroup_descriptives.tables_by_cohort(baseline_cohorts)
clinical_subgroups.to_csv(
    os.path.join(report_outputs_folder, "clinical_subgroup_outcomes.csv"),
    index=False,
)
clinical_subgroups

## Discharge destinations and readmission origins
Additional descriptive reporting requested during peer review; no performance estimation.

In [ ]:
from algo_vs_doc import ward_descriptives

ward_table = ward_descriptives.ward_summary(baseline_cohorts)
ward_table.to_csv(
    os.path.join(report_outputs_folder, "event_characteristics_by_outcome.csv"),
    index=False,
)
ward_table

## Supplementary clinical subgroup outcome table
Descriptive event counts by specialty, admission type, and index ICU length of stay.

In [ ]:
from algo_vs_doc import subgroup_tables

subgroup_table = subgroup_tables.subgroup_summary(clinical_subgroups)
subgroup_table.to_csv(
    os.path.join(report_outputs_folder, "clinical_subgroup_table.csv"),
    index=False,
)
subgroup_table